# Prompt Optimisation and Compression

## Contents
1. Prompt Compression using LLMLingua
2. Prompt Optimisation

## Prompt Compression

Using LLMLingua2

**Step 0**: Install and import the required libraries (`transformers`, `datasets`, `llmlingua`, `torch`, `pandas`)

**Step 1**: Load the text generation model (Llama 3.2) and tokenizer

Model:
- unsloth/Llama-3.2-3B-Instruct



In [ ]:
!pip -q install llmlingua datasets rouge_score

In [ ]:
# import the libraries
import torch
import pandas as pd
from transformers import pipeline
from datasets import load_dataset
from llmlingua import PromptCompressor


In [ ]:
# load the model using pipeline
llm = pipeline(
    "text-generation",
    model="unsloth/Llama-3.2-3B-Instruct",
    dtype=torch.bfloat16,
    device="cuda"
)

Loading weights: 100%|██████████| 254/254 [00:00<00:00, 10259.66it/s]


In [ ]:
# save the tokenzer for later use
tokenizer = llm.tokenizer

**Step 2**: Define helper functions for text generation and token counting

- `generate_answer()`
- `count_tokens()`



    [{"generated_text":                              
        [                                            
            {"role": "system", "content": "..."},    
            {"role": "user", "content": "..."},      
            {"role": "assistant", "content": "..."}  
        ]                                            
    }]

In [ ]:
def generate_answer(prompt):

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    output = llm(messages)

    return output[0]["generated_text"][-1]["content"]

In [ ]:
# define the count_tokens function
def count_tokens(text):
    return len(tokenizer(text, add_special_tokens=False)['input_ids'])

In [ ]:
tokenizer("Prompt Optimisation and Compression", add_special_tokens=False)

{'input_ids': [55715, 31197, 8082, 323, 67261], 'attention_mask': [1, 1, 1, 1, 1]}

In [ ]:
count_tokens("Prompt Optimisation and Compression")

5

**Step 3**: Load the business earnings call dataset

Dataset:
- Aiera/aiera-ect-sum



#### An earnings call transcript is a real conversation between:

Company executives (CEO, CFO, etc.),Financial analysts,Investors

These transcripts are often very long, making them ideal for demonstrating prompt compression.

The dataset contains:

Earnings call transcripts
Human-written summaries

It was created for summarization research.Each example typically contains:

1.transcript

2.summary

In [ ]:
# load the dataset
dataset = load_dataset("Aiera/aiera-ect-sum", split= "test")
dataset

Dataset({
    features: ['aiera_event_id', 'summary', 'transcript'],
    num_rows: 38
})

In [ ]:
# print one row of the dataset
dataset[0]

{'aiera_event_id': 2546776,
 'summary': "Twilio reported strong Q4 2023 financial results, with revenue of $1.076 billion exceeding guidance and growing 5% reported and 8% organically year-over-year. The company demonstrated improved profitability, with non-GAAP income from operations of $173 million, beating expectations due to robust revenue and cost discipline. Twilio also made significant operational progress, signing its largest-ever messaging deal and showcasing its platform's reliability during Cyber Week. Looking ahead, Twilio provided Q1 2024 guidance of $1.025-1.035 billion in revenue and $120-130 million in non-GAAP income from operations. The company expects to exceed 2023's non-GAAP income for the full year 2024, despite $90 million in bonus expenses. Strategically, Twilio is conducting an operational review of its underperforming Segment business and plans to provide an update in March. The company continues to innovate by integrating AI capabilities into its products, ai

**Step 4**: Select an earnings call transcript for the demonstration

- Choose a transcript of suitable length
- Extract the transcript as the context



In [ ]:
# extract the transcript of  any instance of the dataset
sample =dataset[3]
context = sample['transcript']

In [ ]:
context

"\nJohn T. Williams: Thank you, operator. I'm John T. Williams, Head of Investor Relations. Good afternoon, and thank you for joining us to review Nextdoor's first quarter 2021 financial results. With us on the call today are Nirav Tolia, Executive Chair and incoming Chief Executive Officer; and Matt Anderson, Chief Financial Officer.\nNirav Tolia: Thank you, John T., and good afternoon, everyone. It is an honor to reconnect with you as Nextdoor CEO. Today, I have the same feelings of excitement and possibility as I did when we created this company 14 years ago. Nextdoor has certainly grown a lot since then, but I'm confident that our best times are ahead. We had a productive Q1. But before I get into those details, I'd like to briefly discuss how we're thinking philosophically and practically about taking Nextdoor to the next level. In Silicon Valley, there's a commonly held belief that companies benefit when their founders return. I believe this is due to the value of the founder's m

In [ ]:
# print the number of characters of the transcript using the len() function
print(len(context))

9931


**Step 5**: Create the prompt

- Define the summarization instruction
- Combine the transcript and the instruction to form the original prompt
- Count the original prompt tokens



In [ ]:
question = """
Summarize the earnings call in three bullet points.

1. Revenue reported.
2. Key business drivers.
3. Future guidance.
"""

original_prompt = context + "\n\n" + question


In [ ]:
# create the original prompt using the transcript and question
original_prompt

"\nJohn T. Williams: Thank you, operator. I'm John T. Williams, Head of Investor Relations. Good afternoon, and thank you for joining us to review Nextdoor's first quarter 2021 financial results. With us on the call today are Nirav Tolia, Executive Chair and incoming Chief Executive Officer; and Matt Anderson, Chief Financial Officer.\nNirav Tolia: Thank you, John T., and good afternoon, everyone. It is an honor to reconnect with you as Nextdoor CEO. Today, I have the same feelings of excitement and possibility as I did when we created this company 14 years ago. Nextdoor has certainly grown a lot since then, but I'm confident that our best times are ahead. We had a productive Q1. But before I get into those details, I'd like to briefly discuss how we're thinking philosophically and practically about taking Nextdoor to the next level. In Silicon Valley, there's a commonly held belief that companies benefit when their founders return. I believe this is due to the value of the founder's m

In [ ]:
# print the number of tokens of the original prompt
original_prompt_tokens = count_tokens(original_prompt)

In [ ]:
original_prompt_tokens

2003

**Step 6:** Generate a summary using the original prompt

- Pass the original prompt to Llama.
- Display the generated summary.

In [ ]:
# generate an initial answer to the original prompt using the generate_answer function
answer = generate_answer(original_prompt)

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


In [ ]:
# print the initial answer to the original Prompt
answer

'Here are three bullet points summarizing the earnings call:\n\n• Revenue: Nextdoor reported Q1 revenue of $53 million, a 7% year-over-year growth, with organic verified neighbor growth reaching a new high, and weekly active users (WOW) growing to 43.4 million, up 2% year-over-year and 4% sequentially.\n\n• Key business drivers: The company highlighted several key drivers of growth, including continued strong growth in users coming to the platform for the first time, increased engagement, and revenue growth from self-serve customers using the advertising platform, which contributed nearly 50% of total revenue in Q1.\n\n• Future guidance: Nextdoor provided full-year 2024 guidance, expecting revenue to be between $229 million and $235 million, with adjusted EBITDA margin expected to improve by approximately 15 percentage points year-over-year, and positive free cash flow in Q4, 12 months sooner than previously expected.'

**Step 7**: Load the LLMLingua prompt compression model

Model:
- microsoft/llmlingua-2-xlm-roberta-large-meetingbank



In [ ]:
# load the model using PromptCompressor class
compressor = PromptCompressor(model_name ="microsoft/llmlingua-2-xlm-roberta-large-meetingbank", use_llmlingua2=True)


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 978.87it/s] 


In [ ]:
compressor

**Step 8**: Compress the transcript using LLMLingua

- Compress the context
- Create the compressed prompt by appending the original instruction



In [ ]:
# Compress the transcript
compressed = compressor.compress_prompt(context, rate=0.3)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2209 > 512). Running this sequence through the model will result in indexing errors


In [ ]:
# show what the compress_prompt method returns
compressed

{'compressed_prompt': "John T. Williams Head of Investor Relations Nextdoor's first quarter 2021 financial results Nirav Tolia Executive Chair Chief Executive Officer Matt Anderson Chief Financial Officer Nirav Tolia honor reconnect Nextdoor CEO excitement possibility 14 years ago Nextdoor grown best times ahead productive Q1 Nextdoor next level Silicon Valley companies benefit founders return founder's mentality Suh Allen founders undervalued business success mindset instill in Nextdoor technologists value from innovative product potential current implementation not fix founders mentality unwavering focus details ownership accountability for results Innovation starts building community build engaged consumer audience attracts customers robust financial results virtuous circle potential business not easy overnight formula successfulenabled creation Nextdoor excited innovative products Q1 promising start momentum organic growth high Weekly active users 43.4 million up 2% year-over-year 

In [ ]:
#create the compressed_prompt
compressed_p = compressed['compressed_prompt']
compressed_prompt = compressed_p + "\n\n" + question

In [ ]:
compressed_prompt

"John T. Williams Head of Investor Relations Nextdoor's first quarter 2021 financial results Nirav Tolia Executive Chair Chief Executive Officer Matt Anderson Chief Financial Officer Nirav Tolia honor reconnect Nextdoor CEO excitement possibility 14 years ago Nextdoor grown best times ahead productive Q1 Nextdoor next level Silicon Valley companies benefit founders return founder's mentality Suh Allen founders undervalued business success mindset instill in Nextdoor technologists value from innovative product potential current implementation not fix founders mentality unwavering focus details ownership accountability for results Innovation starts building community build engaged consumer audience attracts customers robust financial results virtuous circle potential business not easy overnight formula successfulenabled creation Nextdoor excited innovative products Q1 promising start momentum organic growth high Weekly active users 43.4 million up 2% year-over-year 4% sequentially engage

**Step 9**: Compare the original and compressed prompts

- Count the original tokens
- Count the compressed tokens
- Calculate the compression percentage



In [ ]:
# count the tokens of both the original and the compressed prompts
original_tokens= count_tokens(original_prompt)
compressed_tokens= count_tokens(compressed_prompt)

In [ ]:
print(f"Original prompt tokens: {original_tokens}")
print(f"Compressed prompt tokens: {compressed_tokens}")

Original prompt tokens: 2003
Compressed prompt tokens: 606


In [ ]:
print(
    "Reduction         :",
    f"{(1-compressed_tokens/original_tokens)*100:.2f}%"
)

Reduction         : 69.75%


**Step 10**: Generate a summary using the compressed prompt

- Pass the compressed prompt to Llama
- Display the generated summary



In [ ]:
# generate a compressed_answer of the compressed prompt using the generatae_answer function

compressed_answer =  generate_answer(compressed_prompt)

In [ ]:
# print the compressed_answer to the compressed prompt
compressed_answer

'Here are three bullet points summarizing the earnings call:\n\n• Revenue: Nextdoor reported Q1 revenue of $53 million, a 7% year-over-year growth and a 4% sequential growth, with self-serve customers contributing 50% to the revenue.\n\n• Key business drivers: The company saw strong engagement with a 36% year-over-year increase in sessions, a 4% increase in ARPU, and improved ad delivery performance through its Nextdoor Ads Manager, which also saw revenue growth and increased advertiser spending.\n\n• Future guidance: The company provided a positive outlook, with a 17% improvement in EBITDA margin, full-year EBITDA guidance, and positive free cash flow, while also announcing a repurchase program and plans to reduce costs and improve productivity, indicating a promising start to the year with "best times ahead" for the company.'

**Step 11**: Compare the outputs

- Original prompt length
- Compressed prompt length
- Compression ratio
- Original summary
- Compressed summary
- Observe whether prompt compression preserves the important information while reducing the number of tokens.

In [ ]:
# extarct the ground truth summary
human_summary = sample['summary']

In [ ]:
# print the ground truth summary
human_summary

'Nextdoor reported Q1 revenue of $53 million, a 7% year-over-year increase driven by strong growth in new verified neighbors joining the platform. Adjusted EBITDA margin improved by 17 percentage points year-over-year, reflecting efficiencies in platform costs, marketing spending, and personnel costs. The company\'s advertising platform showed progress, with self-serve contributing nearly 50% of total Q1 revenue. Looking ahead, Nextdoor expects full-year 2024 revenue between $229-235 million and an adjusted EBITDA margin improvement of approximately 15 percentage points year-over-year. The company raised its adjusted EBITDA guidance and now anticipates generating positive free cash flow in Q4 2024, a year ahead of previous projections. Q2 guidance is for approximately $58 million in revenue and a $13 million adjusted EBITDA loss. Nextdoor is focusing on instilling a "founder\'s mentality" to drive innovation and improvements to its core product, with investments being made in AI capabi

In [ ]:
from rouge_score import rouge_scorer
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

# Original answer compared with itself (reference)
original_rouge = scorer.score(answer,human_summary)["rougeL"].fmeasure

# Compressed answer compared with original answer
compressed_rouge = scorer.score(compressed_answer, human_summary)["rougeL"].fmeasure
comparison = pd.DataFrame({

    "Method": [
        "Original Prompt",
        "LLMLingua"
    ],

    "Prompt Tokens": [
        original_tokens,
        compressed_tokens
    ],

    "Generated Answer": [
        answer,
        compressed_answer
    ],

    "ROUGE-L Score": [
        round(original_rouge, 4),
        round(compressed_rouge, 4)
    ]

})

comparison

,Method,Prompt Tokens,Generated Answer,ROUGE-L Score
0,Original Prompt,2003,Here are three bullet points summarizing the e...,0.3193
1,LLMLingua,606,Here are three bullet points summarizing the e...,0.2442


## Loading LLM and defining utility functions

In [ ]:
# load the model if not already done


    [{"generated_text":                              
        [                                            
            {"role": "system", "content": "..."},    
            {"role": "user", "content": "..."},      
            {"role": "assistant", "content": "..."}  
        ]                                            
    }]                                               

In [ ]:
# ============================================================
# Utility Functions
# ============================================================

def make_message(system_content, user_content):
    return [
        {"role": "system", "content": system_content},
        {"role": "user", "content": user_content},
    ]


def fetch_response(response):
    return response[0]["generated_text"][-1]["content"]



In [ ]:

# define the generate function to generate a response from the model given a system prompt and a user prompt
def generate(system_prompt, user_prompt, max_new_tokens=300):
    messages =make_message(system_prompt, user_prompt)

    responnses = llm(messages, max_new_tokens= max_new_tokens, do_sample=False)

    return fetch_response(responnses)


## Prompt Optmization

We will use the **ProTeGi** automatic prompt optimization algorithm for this demonstration. ProTeGi uses textual gradients to find flaws in the initial prompt and uses another (or same) LLM to create optimized version of the prompt.

**Scenario**: An application for a loan has been received by the bank. They have to decide whether to accept or reject the loan based upon certain lending criteria and parameters about the applicant.

It is difficult to manually write a prompt to automate this process. It is also difficult to manually optimize a simple prompt for such a complex scenario. Automatic prompt optimization is useful in this case to derive an optimized prompt starting from a simple prompt.

### ProTeGi: Gradient-Based Prompt Optimization Workflow

#### **Step 0:** Background info


##### Credit Policy

In [ ]:
credit_policy = """
NORTHBRIDGE BANK - RETAIL LENDING POLICY

P1. Debt-to-Income Ratio (DTI) = Existing Debt / Annual Income.
    DTI must be less than 0.40.

P2. Loan-to-Income Ratio (LTI) = Requested Loan / Annual Income.
    LTI must not exceed 3.0.

P3. Minimum credit score is 660.
    Scores between 620 and 659 are acceptable ONLY if collateral is provided.

P4. Applicants must have at least 12 months of continuous employment.

P5. Unsecured loans above $100,000 are not permitted.

P6. Final decision must be one of:
    APPROVE
    REJECT
    CONDITIONAL APPROVAL
"""


##### Loan Application

In [ ]:
loan_application = """
Applicant Name: John Smith

Age: 28

Annual Income: $48,000

Employment:
Software Developer at ABC Technologies

Employment Duration:
10 months

Credit Score:
615

Existing Debt:
$22,000

Requested Loan Amount:
$180,000

Loan Purpose:
Purchase a house

Collateral:
None

Previous Loan Defaults:
None
"""


##### Ground Truth

For this applicant,\
DTI = 22000 / 48000\
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;= 0.458 (violate P1)

LTI = 180000 / 48000\
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;= 3.75 (violate P2)

Applicant Score is 615 with no collateral (violate P3)

Employment Duration is 10 months (violate P4)

Applicant request of loan 180000 with no collateral (violate p5)

**Correct decision is reject**

In [ ]:
correct_decision = "reject"

#### **Step 1:** Define the Initial Prompt

##### The initial prompt

In [ ]:
system_prompt = """
You are a bank loan officer.

Review the loan application and decide whether the loan should be approved.

Provide a brief explanation.
"""

user_prompt = f"""
BANK LENDING POLICY

{credit_policy}

----------------------------------------

LOAN APPLICATION

{loan_application}
"""


#### **Step 2:** Generate the Initial Response

In [ ]:
#  generate the inital answer
initial_answer = generate(system_prompt, user_prompt)

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [ ]:
# print the initial answer
initial_answer

"Based on the provided loan application and Northbridge Bank's retail lending policy, I am going to approve the loan with a conditional approval.\n\nHere's the explanation:\n\n- The applicant's debt-to-income ratio (DTI) is $22,000 / $48,000 = 0.46, which exceeds the maximum allowed limit of 0.40. However, since the applicant is purchasing a house, which is a secured loan, the DTI ratio is not a major concern.\n\n- The loan-to-income ratio (LTI) is $180,000 / $48,000 = 3.75, which exceeds the maximum allowed limit of 3.0. This is a significant concern, but considering the applicant is purchasing a house, the LTI ratio is not a major concern.\n\n- The applicant's credit score is 615, which falls within the acceptable range of 620-659 when collateral is provided. Since the applicant is not providing collateral, the credit score is a concern.\n\n- The applicant has less than 12 months of continuous employment, which does not meet the minimum requirement of 12 months.\n\n- The requested lo

##### **Analysis of the Initial Response**



The final decision is not consistent with the bank's lending policy. The following issues can be observed:

Mistakes in the Model's Response
- Incorrect Final Decision
- Incorrect Credit Score Interpretation

However, after identifying these policy violations, it still concludes with Conditional Approval, making the reasoning internally inconsistent.

#### **Step 3:** Evaluate the Response
- Use an **LLM-as-a-Judge** (can be same or different model) to evaluate the generated response.
- The judge assigns a score and identifies:
  - Correct reasoning
  - Missing information
  - Errors or hallucinations
  - Policy violations

##### LLM Judge Prompt

In [ ]:
judge_system_prompt = """
You are an expert loan approval auditor.

Evaluate ONLY the quality of the model's reasoning.
Use the following rubric (10 points).
1. Correctly evaluates the credit score. (1 point)
2. Correctly evaluates annual income. (1 point)
3. Correctly evaluates existing debt. (1 point)
4. Correctly evaluates employment stability. (1 point)
5. Correctly evaluates collateral. (1 point)
6. Computes or discusses the Debt-to-Income (DTI) ratio. (1 point)
7. Computes or discusses the Loan-to-Income (LTI) ratio. (1 point)
8. References the lending policy when making the decision. (1 point)
9. Gives a clear evidence-based justification. (1 point)
10. Correct final decision. (1 point)

Be STRICT.
Only award a point if the criterion is explicitly satisfied.
A generic explanation should score between 3 and 5.
A detailed policy-based analysis should score between 8 and 10.

Return EXACTLY in this format:

Score: X/10
Strengths:
- ...
Missing:
- ...
"""

judge_user_prompt = f"""
BANK POLICY

{credit_policy}

----------------------------------------

LOAN APPLICATION

{loan_application}

----------------------------------------

MODEL RESPONSE

{initial_answer}

----------------------------------------

ACTUAL FINAL DECISION

{correct_decision}
"""

In [ ]:
# generate judge feedback
judge_feedback = generate(judge_system_prompt, judge_user_prompt)

In [ ]:
# print Judge Feedback
judge_feedback

### **Step 4:** Generate a Textual Gradient
Generate a **textual gradient**, i.e., natural-language feedback describing how the prompt should be improved.

##### Gradient Prompt

In [ ]:
gradient_system_prompt = """
Your task is to find out the textual gradients of a prompt. The textual gradient is the analysis of why the CURRENT PROMPT produced a weak response.

Do NOT rewrite the prompt.

Instead, identify the instructions that are missing from the prompt.

Focus on:

- Missing reasoning steps
- Missing financial analyses
- Missing policy references
- Missing output structure
- Missing justification requirements

Write 4-6 concise bullet points.

Each bullet should explain:

- what the prompt failed to instruct
- why that caused a weaker answer

Return ONLY the textual gradient.
"""

gradient_user_prompt = f"""
CURRENT PROMPT

{system_prompt}

--------------------------------------------------

BANK POLICY

{credit_policy}

--------------------------------------------------

LOAN APPLICATION

{loan_application}

--------------------------------------------------

MODEL RESPONSE

{initial_answer}

--------------------------------------------------

JUDGE FEEDBACK

{judge_feedback}

Generate the textual gradient.
"""

In [ ]:
# Generate Textual Gradient
textual_gradient = generate(gradient_system_prompt, gradient_user_prompt)

In [ ]:
# Print the textual gradient
textual_gradient

### **Step 5:** Optimize the Prompt

##### Optimizer Prompt

In [ ]:
optimizer_system_prompt = """
You are an expert Prompt Engineer. Your task is to improve the prompt using ONLY the textual gradient.

Requirements:

- Preserve the original task.
- Incorporate the feedback from the textual gradient.
- Add only the missing instructions.
- Keep the prompt concise and professional.
- Do not include explanations or reasoning outside the prompt.
- Return ONLY the improved prompt.
"""

optimizer_user_prompt = f"""
ORIGINAL PROMPT

{system_prompt}

--------------------------------------------------

TEXTUAL GRADIENT

{textual_gradient}

--------------------------------------------------

Rewrite the prompt by incorporating the missing instructions identified in the textual gradient.
"""

In [ ]:
# generate the optimised prompt
optimised_prompt = generate(optimizer_system_prompt, optimizer_user_prompt)

In [ ]:
# print the optimised prompt
optmised_prompt

Analysis of the Optimized Prompt

Compared to the original prompt, the optimized prompt incorporates several important instructions identified through the textual gradient:

- Explicitly requires financial calculations
- Grounds the decision in the bank policy
- Improves justification quality
- Provides clearer decision instructions

### **Step 6:** Generate the Optimized Response
Generate a new response using the optimized prompt for the same input.



In [ ]:
# generate the optimised answer
optmised_answer = generate(system_prompt, optimised_prompt)

In [ ]:
# print the optimised answer
optimised_answer

### **Step 7:** Re-evaluate the Optimized Response
- Evaluate the optimized response using the same judge and evaluation criteria.
- Assign a new score to measure the improvement.



##### optimised judge user prompt

In [ ]:
optimized_judge_user_prompt = f"""
BANK POLICY

{credit_policy}

----------------------------------------

LOAN APPLICATION

{loan_application}

----------------------------------------

MODEL RESPONSE

{optimised_answer}

----------------------------------------

ACTUAL FINAL DECISION

{correct_decision}
"""

In [ ]:
# generate the optimised judge feedback
optimised_judge_feedback = generate(judge_system_prompt, optimized_judge_user_prompt)

In [ ]:
# print the optimised judge feedback
optimised_judge_feedback

### Analysis of the initial prompt

The initial prompt doesn't tell the model to:

  - calculate DTI, LTI
  - not justify policy violations
  - verify every policy rule
  - explain which rules passed or failed
  - justify the decision with evidence
  - use the allowed output labels exactly

The prompt is too generic.